# Cloud Security & DevOps Automation Pipeline in Google Cloud Platform (GCP)
This notebook demonstrates a combined DevOps and Cloud Security workflow using Python:
1. Programmatically managing/verifying Google Cloud Platform (GCP) projects and authentication.
2. Automating security compliance checks (scanning IAM bindings for overly permissive access).
3. Remediation reporting.

In [1]:
!pip install google-cloud-resource-manager
!pip install google-cloud-asset

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [google-cloud-asset]


---

#### ⚠️ Do not forget to "RESTART RUNTIME" or "RESTART KERNEL"

---

In [1]:
import os
import subprocess
from google.cloud import resourcemanager_v3
from google.cloud import asset_v1

In [2]:
from google.colab import auth
# Authenticate the user to access Google Cloud resources
auth.authenticate_user()
print("Successfully authenticated with Google Cloud!")

Successfully authenticated with Google Cloud!


In [3]:
# 1. Dynamic Project Setup via CLI/Environment
# If you don't have a project yet, you can create one via CLI:
!gcloud projects create my-secops-demo-project-2026 --name="SecOps Demo" || true

# Configure gcloud to use our newly created project
!gcloud config set project my-secops-demo-project-2026

# Automatically fetch the active project ID without hardcoding
PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT")

if not PROJECT_ID:
    # Fallback to grab from active gcloud config if running locally/Colab
    result = subprocess.run(["gcloud", "config", "get-value", "project"], capture_output=True, text=True)
    PROJECT_ID = result.stdout.strip()

print(f"Active Google Cloud Project ID: {PROJECT_ID}")

ERROR: (gcloud.projects.create) Project creation failed. The project ID you specified is already in use by another project. Please try an alternative ID.
Updated property [core/project].
Active Google Cloud Project ID: my-secops-demo-project-2026


### Step 1: DevOps Resource Verification & API Enablement
Before running security scans, we must ensure required security and resource manager APIs are active.

In [4]:
# DevOps automated check & enablement of security-critical APIs
required_apis = [
    "cloudresourcemanager.googleapis.com",
    "iam.googleapis.com",
    "cloudasset.googleapis.com"
]

In [5]:
print("Verifying and enabling required GCP APIs...")
for api in required_apis:
    subprocess.run(["gcloud", "services", "enable", api, f"--project={PROJECT_ID}"], check=True)
print("All required APIs are enabled successfully.")

Verifying and enabling required GCP APIs...
All required APIs are enabled successfully.


### Step 2: Cloud Security Policy Scanner
We will check the project's IAM policy for security misconfigurations—specifically looking for public exposure (`allUsers` or `allAuthenticatedUsers`) or dangerous primitive roles (`roles/owner`).

In [6]:
import google.auth
from googleapiclient import discovery

In [7]:
# Authenticate and build the Cloud Resource Manager client
credentials, _ = google.auth.default()
crm_service = discovery.build('cloudresourcemanager', 'v1', credentials=credentials)

In [8]:
def audit_iam_policies(project_id):
    try:
        policy = crm_service.projects().getIamPolicy(resource=project_id).execute()
        bindings = policy.get('bindings', [])

        security_findings = []
        print(f"Scanning IAM bindings for project: {project_id}...\n")

        for binding in bindings:
            role = binding.get('role')
            members = binding.get('members', [])

            # Security Rule Check 1: Public exposure
            for member in members:
                if member in ["allUsers", "allAuthenticatedUsers"]:
                    security_findings.append(f"[CRITICAL] Public Exposure Risk: Role '{role}' is granted to '{member}'.")
                    print(f"‘[CRITICAL] Public Exposure: '{role}' given to '{member}'")

            # Security Rule Check 2: Primitive Roles usage
            if role in ["roles/owner", "roles/editor"]:
                print(f"☑ [WARNING] Broad Primitive Role: '{role}' assigned to {members}")

        if not security_findings:
            print("✅ No critical public exposure vulnerabilities detected in IAM bindings.")

    except Exception as e:
        print(f"Error fetching IAM policy: {e}")

In [9]:
audit_iam_policies(PROJECT_ID)

Scanning IAM bindings for project: my-secops-demo-project-2026...

☑ [WARNING] Broad Primitive Role: 'roles/owner' assigned to ['user:rguin26@gmail.com']
✅ No critical public exposure vulnerabilities detected in IAM bindings.


### Pipeline Complete
The security script successfully evaluated the project topology dynamically without relying on hardcoded strings.